# Lesson 01 · Microsoft AutoGen · All 3 Demos in One Notebook

**One company runs through every demo: PaySetu**, a (fictional) Mumbai fintech with 400 employees. It runs a UPI payments app and a finance team that files GST e-invoices.

| Demo | What trainees see | Real-world link |
|---|---|---|
| **1️⃣ Demo 1** · First AutoGen call and first agent | One question to the AI, then an HR helpdesk agent answering an employee about maternity leave | Every company HR helpdesk |
| **2️⃣ Demo 2** · A compliance team of 3 agents | Agents review PaySetu's real-looking policies against Indian rules (Maternity Benefit Act, POSH Act, CERT-In, RBI) and route the fixes | Internal audit, compliance, HR, IT security |
| **3️⃣ Demo 3** · Agents agreeing on a "format" | PaySetu's accounts system speaks XML, the GST e-invoice portal speaks JSON; the agents negotiate, convert, or fail safely | Tally/ERP exports, GST e-invoicing, bank file formats |

**How to avoid mixing up cells:** every code cell starts with its tag:
`# 🅰️ SETUP` (run once) · `# 1️⃣ DEMO 1` · `# 2️⃣ DEMO 2` · `# 3️⃣ DEMO 3`

**What changed from the original demo files:**
- One notebook instead of 2 scripts + 3 documents.
- One version of AutoGen everywhere (the current `autogen-agentchat`). The old Demo 2 mixed the old `autogen`/`ag2` package with the new one.
- The API key is read from `.env`, never written in the code. ⚠️ The original files had the key written in them: please ask for it to be rotated.
- Demo 2's summary is now written by the agents. The original printed a fixed, pre-written "Key Findings" list whatever the agents said.
- Unused code (webhook and custom-rules functions that were never called) is removed.
- Generic examples ("Quarterly Report", "Remote Work Policy") are replaced with real Indian rules and formats.

---
# 🅰️ SETUP (run once, before class)

## Setup 1 · Install
Installs AutoGen's agent library, its Azure OpenAI connector, and `python-dotenv` (reads the key from `.env`).

*Outside Jupyter (the original Demo 1 steps):* `mkdir demo1 && cd demo1`, `python3 -m venv venv`, `source venv/bin/activate`, then the same `pip install` without the `%`.

In [1]:
# 🅰️ SETUP 1 · Install (run once)
%pip install -q -U autogen-agentchat "autogen-ext[openai]" python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.75.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Setup 2 · Connect to gpt-5-mini through the company gateway
**What it does:** creates the `model_client`, the "brain" every agent in this notebook will use, and sends a one-word test.

**Notes for you:**
- Put the key in a `.env` file next to this notebook: `AZURE_OPENAI_API_KEY=...`
- This uses the same gateway route as your other notebooks (`/openaiprodtest-clone/...`, api-version `2024-12-01-preview`).
- The original course files used a different route: `azure_endpoint=GATEWAY` with `api_version="2024-10-21"`. If your lab only works with that one, swap the two commented lines in.
- No `temperature` is set: gpt-5-mini only accepts the default.

In [1]:
# 🅰️ SETUP 2 · Connect to gpt-5-mini through the Azure gateway
import os, warnings
warnings.filterwarnings("ignore", message="Resolved model mismatch")   # harmless version-name notice
from dotenv import load_dotenv, find_dotenv
from autogen_core.models import UserMessage
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

load_dotenv(find_dotenv(usecwd=True), override=True)
api_key="2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"

GATEWAY = os.getenv("AZURE_GATEWAY_URL", "https://openai-api-management-gw.azure-api.net")

model_client = AzureOpenAIChatCompletionClient(
    base_url=f"{GATEWAY}/openaiprodtest-clone",   # alternative: azure_endpoint=GATEWAY,
    api_version="2024-12-01-preview",             # alternative: api_version="2024-10-21",
    azure_deployment="gpt-5-mini",
    model="gpt-5-mini",
    api_key=api_key,
)

test = await model_client.create([UserMessage(content="Reply with just the word: ready", source="user")])
print("Model says:", test.content)

Model says: ready


---
# 1️⃣ DEMO 1 · Your first AutoGen call, then your first agent

**Say before starting:** *"Before we build a team, let's meet one team member. First we ask the AI a single question. Then we give it a job title and turn it into an agent."*

## Demo 1 · Step 1 · One direct question to the model
**What it does:** sends one message straight to the model. No agent yet, just question in, answer out.

`await` simply means "wait for the answer to come back". Jupyter allows it directly in a cell.

**Point out:** this is like calling a very knowledgeable friend. Useful, but they don't have a job, rules, or memory of who they work for.

In [2]:
# 1️⃣ DEMO 1 · Step 1 · One direct question to the model
response = await model_client.create([
    UserMessage(content="In 3 short lines, explain Microsoft AutoGen to an HR manager who has never coded.", source="user")
])
print(response.content)

Microsoft AutoGen is a platform that sets up and coordinates multiple AI assistants to work together.  
It automates HR tasks—resume screening, interview scheduling, and employee Q&A—by assigning clear roles and steps.  
Enterprise-ready: it connects securely to your systems and can be customized without heavy coding.


## Demo 1 · Step 2 · Turn it into an agent: PaySetu's HR helpdesk
**What it does:** `AssistantAgent` wraps the model with a **name** and a **system message**, which is its job description. Now it behaves like PaySetu's HR helpdesk, not a general chatbot.

**Real-world question:** an employee asks how much paid maternity leave she gets. In India, the Maternity Benefit (Amendment) Act, 2017 raised it from 12 to **26 weeks** for the first two children.

**Say:** *"Remember this answer. In Demo 2 we'll find that PaySetu's own policy document still says 12 weeks."*

In [3]:
# 1️⃣ DEMO 1 · Step 2 · First agent: PaySetu's HR helpdesk
from autogen_agentchat.agents import AssistantAgent

hr_helpdesk = AssistantAgent(
    name="HR_Helpdesk",
    model_client=model_client,
    system_message=(
        "You are the HR helpdesk for PaySetu, a 400-employee fintech in Mumbai. "
        "Answer employees warmly in under 80 words. Mention the Indian law your answer is based on. "
        "If you are not sure, say HR will confirm by email."
    ),
)

result = await hr_helpdesk.run(
    task="Hi, I joined PaySetu last year and I'm expecting my first baby in March. How many weeks of paid maternity leave can I take?"
)
print(result.messages[-1].content)

Congratulations! Under the Maternity Benefit Act, 1961 (as amended 2017), you are entitled to 26 weeks paid maternity leave for your first (and second) child, provided you have worked at least 80 days in the 12 months before your expected delivery. HR will confirm your eligibility and company process by email.


### 1️⃣ Demo 1 wrap-up (say this)
*"Same model, two very different behaviours. The model alone is a smart stranger. Give it a name and a job description, and it becomes an agent that works for PaySetu. Next: three agents working as a team."*

**Try in class:** ask the room for HR questions (notice period, gratuity after 5 years, leave encashment) and re-run Step 2 with them.

---
# 2️⃣ DEMO 2 · A compliance team of 3 agents reviews PaySetu's policies

**The real-world problem (read this out):** companies keep hundreds of policy documents. Laws change, and nobody re-reads the PDFs. Examples that really catch Indian companies out:
- HR leave policies still saying **12 weeks** of maternity leave, years after the 2017 law made it **26 weeks**.
- IT security SOPs saying incidents are reported "within 72 hours" (a GDPR habit), when **CERT-In** in India has required **6 hours** since 2022.
- Customer refund SOPs promising "7 working days" for failed UPI payments, when **RBI** requires auto-reversal by the **next day (T+1)** and **₹100 per day** compensation after that.

An internal audit team would take weeks to catch all of this. We'll build a team of 3 AI agents that does a first pass in about a minute:

```
📄 3 policy documents + 📕 rulebook
        │
 🔍 DocumentScanner     → lists each document: owner, type, how old it is
        │
 ⚖️ ComplianceChecker    → compares each one with the rulebook: issue, severity, fix
        │
 📬 ReviewCoordinator    → decides who must fix what, by when, then says TERMINATE
```

## Demo 2 · Step 1 · The documents and the rulebook
**What it does:** three PaySetu documents, each with real-world problems hidden inside, and a **rulebook** of actual Indian rules.

**Why a rulebook?** We don't let the AI rely on memory for legal facts. We hand it the rules, the way an auditor carries a checklist. This keeps the answers grounded and checkable.

**Explain in class:** let trainees spot the problems first. Ask: *"Which of these would get PaySetu into trouble?"* Then let the agents check.

| Document | Hidden problems |
|---|---|
| HR-07 Leave Policy | 12 weeks maternity leave (law: 26); harassment complaints go to "the HR manager" (POSH: an Internal Committee); last reviewed 2016 |
| IT-03 Cyber Incident SOP | Report within 72 hours (CERT-In: 6 hours); logs kept 30 days (CERT-In: 180 days) |
| CS-11 Failed UPI Refund SOP | 7 working days (RBI: T+1, then ₹100/day); no approver signature |

In [4]:
# 2️⃣ DEMO 2 · Step 1 · PaySetu's policy documents and the rulebook
POLICY_DOCUMENTS = [
    {"id": "HR-07", "title": "Employee Leave Policy", "owner": "HR", "last_reviewed": "2016-04-01",
     "approved_by": "Meera Iyer, Head of HR",
     "content": """
     4.2 Maternity leave: Women employees get 12 weeks of paid maternity leave.
     4.3 Paternity leave: 5 working days.
     7.1 Complaints of sexual harassment must be reported to the HR Manager, who will investigate.
     """},
    {"id": "IT-03", "title": "Cyber Incident Response SOP", "owner": "IT Security", "last_reviewed": "2021-08-15",
     "approved_by": "Arjun Rao, CISO",
     "content": """
     2.1 Serious incidents (data breach, ransomware, account takeover) are reported to senior management within 72 hours.
     2.2 External authorities are informed if management decides it is required.
     3.4 Server and application logs are kept for 30 days, then deleted.
     """},
    {"id": "CS-11", "title": "Failed UPI Transaction Refund SOP", "owner": "Customer Support", "last_reviewed": "2024-11-10",
     "approved_by": "",
     "content": """
     1. If a customer's UPI payment fails but their account is debited, raise a support ticket.
     2. Refund is processed within 7 working days.
     3. Agents may offer a Rs 50 cashback coupon if the customer is upset.
     """},
]

RULEBOOK = """
R1 Maternity Benefit (Amendment) Act, 2017: 26 weeks of paid maternity leave for a woman's first two children (12 weeks from the third child).
R2 POSH Act, 2013: every employer with 10 or more employees must set up an Internal Committee (IC), presided over by a senior woman employee. Harassment complaints go to the IC, not to a single manager.
R3 CERT-In Directions, 28 April 2022: report specified cyber incidents to CERT-In within 6 hours of noticing them. Keep logs of all ICT systems for a rolling 180 days, within India.
R4 RBI circular on turnaround time for failed transactions, 20 Sept 2019: if a UPI payment fails and the customer is debited but the beneficiary is not credited, auto-reverse by T+1 day. If late, pay the customer Rs 100 per day of delay, without them having to ask.
R5 PaySetu internal rule: every policy must be reviewed at least once every 12 months and must name an approver.
"""

TODAY = "2026-10-17"   # the date the audit is run
print(f"{len(POLICY_DOCUMENTS)} documents and {len([l for l in RULEBOOK.splitlines() if l.startswith('R')])} rules loaded")

3 documents and 5 rules loaded


## Demo 2 · Step 2 · Hire the team (3 agents)
**What it does:** three `AssistantAgent`s, each with a strict job description. Read one aloud: it's written like a job description for a real person.

**Point out:**
- The **ComplianceChecker** must quote the rule number (R1–R5). If it can't point to a rule, it's not allowed to call it an issue. This stops the AI from inventing laws.
- **Severity** is defined in plain words, so two runs give consistent ratings.
- Only the **ReviewCoordinator** can end the meeting, by writing `TERMINATE`.

In [5]:
# 2️⃣ DEMO 2 · Step 2 · Three specialist agents
from autogen_agentchat.agents import AssistantAgent

document_scanner = AssistantAgent(
    name="DocumentScanner",
    model_client=model_client,
    system_message=(
        "You are the Document Scanner at PaySetu. For each document, list in one line: ID, title, owner team, "
        f"last reviewed date, how many months old it is as of {TODAY}, and whether an approver is named. "
        "Do not judge compliance. That is someone else's job. Keep it short."
    ),
)

compliance_checker = AssistantAgent(
    name="ComplianceChecker",
    model_client=model_client,
    system_message=(
        "You are the Compliance Checker at PaySetu. Compare every document ONLY against this rulebook:\n"
        + RULEBOOK +
        "\nFor each problem, give a table row: Document | Rule (R1-R5) | What the document says | What the rule requires | Severity.\n"
        "Severity: CRITICAL = breaks a law or regulator rule; MAJOR = internal rule broken (R5); MINOR = wording or clarity.\n"
        "Only report issues you can tie to a rule number. Do not invent laws."
    ),
)

review_coordinator = AssistantAgent(
    name="ReviewCoordinator",
    model_client=model_client,
    system_message=(
        "You are the Review Coordinator at PaySetu. Using the checker's table, write the action plan:\n"
        "Document | Fix needed | Send to | Deadline\n"
        "Routing: CRITICAL -> Legal and the document owner's head, deadline 7 days; "
        "MAJOR -> document owner's head, 30 days; MINOR -> document owner, 60 days.\n"
        "End with a 3-line summary for the CEO in plain English. Then write the word TERMINATE on its own line."
    ),
)
print("Team hired: DocumentScanner, ComplianceChecker, ReviewCoordinator ✅")

Team hired: DocumentScanner, ComplianceChecker, ReviewCoordinator ✅


## Demo 2 · Step 3 · Put them in a meeting room (the team)
**What it does:**
- `RoundRobinGroupChat` = a meeting where people speak **in turn**: Scanner, then Checker, then Coordinator. Everyone hears everything said before them (shared conversation).
- **Termination** = when the meeting ends: when someone says `TERMINATE`, **or** after 8 messages as a safety net, so it can never run forever (a guardrail).

**Explain in class:** *"Like a weekly review meeting with a fixed agenda: the analyst presents, the auditor comments, the manager assigns actions and closes the meeting."*

In [6]:
# 2️⃣ DEMO 2 · Step 3 · Put the agents in a round-robin meeting with stop rules
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

stop_rules = TextMentionTermination("TERMINATE") | MaxMessageTermination(8)

compliance_team = RoundRobinGroupChat(
    participants=[document_scanner, compliance_checker, review_coordinator],
    termination_condition=stop_rules,
)
print("Meeting room ready ✅")

Meeting room ready ✅


## Demo 2 · Step 4 · Run the review
**What it does:** hands the documents to the team and streams the conversation live. `Console(...)` prints each agent's message as it arrives.

**Point out while it runs:**
1. **Scanner** notices HR-07 hasn't been reviewed in about 10 years, and CS-11 has no approver.
2. **Checker** ties each problem to a rule number: 12 vs. 26 weeks (R1), HR manager vs. Internal Committee (R2), 72 vs. 6 hours and 30 vs. 180 days (R3), 7 days vs. T+1 (R4).
3. **Coordinator** turns it into who-fixes-what-by-when, plus a 3-line CEO summary, then says TERMINATE.

**Ask the room:** *"Which of these would you fix first, and why?"* (Usually R3 and R4: a regulator can act on those quickly.)

In [7]:
# 2️⃣ DEMO 2 · Step 4 · Run the compliance review (streams live)
from autogen_agentchat.ui import Console

def describe(d):
    return (f"Document {d['id']}: {d['title']}\nOwner: {d['owner']} | Last reviewed: {d['last_reviewed']} | "
            f"Approved by: {d['approved_by'] or '[MISSING]'}\nContent:{d['content']}")

documents_text = "\n\n".join(describe(d) for d in POLICY_DOCUMENTS)

await compliance_team.reset()     # start with an empty meeting room every time
review = await Console(compliance_team.run_stream(
    task=f"Today is {TODAY}. Review these PaySetu documents for compliance:\n\n{documents_text}"
))
print("\nStopped because:", review.stop_reason)

---------- TextMessage (user) ----------
Today is 2026-10-17. Review these PaySetu documents for compliance:

Document HR-07: Employee Leave Policy
Owner: HR | Last reviewed: 2016-04-01 | Approved by: Meera Iyer, Head of HR
Content:
     4.2 Maternity leave: Women employees get 12 weeks of paid maternity leave.
     4.3 Paternity leave: 5 working days.
     7.1 Complaints of sexual harassment must be reported to the HR Manager, who will investigate.
     

Document IT-03: Cyber Incident Response SOP
Owner: IT Security | Last reviewed: 2021-08-15 | Approved by: Arjun Rao, CISO
Content:
     2.1 Serious incidents (data breach, ransomware, account takeover) are reported to senior management within 72 hours.
     2.2 External authorities are informed if management decides it is required.
     3.4 Server and application logs are kept for 30 days, then deleted.
     

Document CS-11: Failed UPI Transaction Refund SOP
Owner: Customer Support | Last reviewed: 2024-11-10 | Approved by: [MISSING

## Demo 2 · Step 5 · Save the action plan as a report
**What it does:** takes the Coordinator's final message and saves it as `PaySetu_Compliance_Review.md`, a file you could email to the CEO or upload to Jira.

**Say:** *"This is the difference between a demo and a real tool: the output is something a person can act on tomorrow morning."*

In [8]:
# 2️⃣ DEMO 2 · Step 5 · Save the Coordinator's action plan as a report file
plan = [m.content for m in review.messages if getattr(m, "source", "") == "ReviewCoordinator"]
report = plan[-1].replace("TERMINATE", "").strip() if plan else "No action plan was produced."

with open("PaySetu_Compliance_Review.md", "w", encoding="utf-8") as f:
    f.write(f"# PaySetu Policy Compliance Review ({TODAY})\n\n{report}\n")

print(report)
print("\n📄 Saved to PaySetu_Compliance_Review.md")

Document | Fix needed | Send to | Deadline
HR-07 (Employee Leave Policy) | Revise maternity leave to comply with R1: 26 weeks paid leave for a woman's first two children (12 weeks from third child). Update payroll procedures and communicate retroactive/ongoing entitlements. | Legal; Head of HR (Meera Iyer) | 2026-10-24 (7 days)
HR-07 (Employee Leave Policy) | Revise POSH wording to comply with R2: remove requirement to report complaints to a single HR Manager; establish and document an Internal Committee (IC) per law (constitution, senior woman chair, complaint intake, timelines, training, confidentiality). | Legal; Head of HR (Meera Iyer) | 2026-10-24 (7 days)
HR-07 (Employee Leave Policy) | Bring policy up-to-date and re-approve: perform full review, update "last reviewed" date, version control, and schedule annual reviews. | Head of HR (Meera Iyer) | 2026-11-16 (30 days)

IT-03 (Cyber Incident Response SOP) | Mandate CERT-In reporting per R3: update SOP so specified incidents (data 

## Demo 2 · Step 6 (optional) · Let an AI chairperson pick who speaks next
**What it does:** `SelectorGroupChat` replaces the fixed order with a chairperson (the model) who reads the conversation and picks the next speaker. This is what the original demo's `GroupChatManager` did.

**When it matters:** fixed order is predictable and cheaper. A chairperson is better when the order depends on what comes up, for example sending a document back to the Scanner if the Checker finds missing information.

**Say:** *"Round-robin is a fixed agenda. Selector is a chairperson who decides who speaks next."*

In [9]:
# 2️⃣ DEMO 2 · Step 6 (optional) · Same team, but an AI chairperson picks the next speaker
from autogen_agentchat.teams import SelectorGroupChat

chaired_team = SelectorGroupChat(
    participants=[document_scanner, compliance_checker, review_coordinator],
    model_client=model_client,
    termination_condition=TextMentionTermination("TERMINATE") | MaxMessageTermination(8),
)
_ = await Console(chaired_team.run_stream(
    task=f"Today is {TODAY}. Review only this document for compliance:\n\n" + describe(POLICY_DOCUMENTS[2])
))

---------- TextMessage (user) ----------
Today is 2026-10-17. Review only this document for compliance:

Document CS-11: Failed UPI Transaction Refund SOP
Owner: Customer Support | Last reviewed: 2024-11-10 | Approved by: [MISSING]
Content:
     1. If a customer's UPI payment fails but their account is debited, raise a support ticket.
     2. Refund is processed within 7 working days.
     3. Agents may offer a Rs 50 cashback coupon if the customer is upset.
     
---------- TextMessage (ComplianceChecker) ----------
Document | Rule (R1-R5) | What the document says | What the rule requires | Severity
CS-11 | R4 | "1. If a customer's UPI payment fails but their account is debited, raise a support ticket. 2. Refund is processed within 7 working days. 3. Agents may offer a Rs 50 cashback coupon if the customer is upset." | R4 requires auto-reverse by T+1 day when a UPI payment fails and the customer is debited; if reversal is late, pay the customer Rs 100 per day of delay, without them ha

### 2️⃣ Demo 2 wrap-up (say this)
*"Three agents, three job descriptions, one shared meeting, and a clear rule for ending it. The rulebook kept them honest: every issue had to point to a real rule. In a real company, you'd point this at the policy folder every quarter, and the compliance team reviews the agents' findings instead of reading every PDF."*

**Important honesty point for trainees:** the agents' output is a first pass. A human (legal or compliance) still signs off. That's human-in-the-loop.

---
# 3️⃣ DEMO 3 · Agents agreeing on a "format" before they talk

**The real-world problem (read this out):** PaySetu's finance team uses an accounting system that exports invoices as **XML** (Tally, for example, exchanges data in XML). The **GST e-invoice portal (IRP)** only accepts **JSON**. Two systems, two "languages" for the same invoice.

Agents have the same problem. Before sending, they must agree on a format:

```
1. DECLARE    each agent lists the formats it can read and write
2. NEGOTIATE  find a format both share, in order of preference
3. CONVERT    if the sender's file isn't in the agreed format, translate it
4. DELIVER    send it, or FAIL SAFELY with a clear log if there's no common format
```

**Everyday comparison:** a plug and a socket. If they match, plug in. If you have a travel adapter, convert. If not, don't force it into the socket: stop and report it.

This demo is plain Python (no AI calls), so it's instant and always gives the same result.

## Demo 3 · Step 1 · Translators between XML and JSON
**What it does:** four small helpers that turn an invoice into XML or JSON and back. Trainees don't need the details: *"These are the travel adapters."*

In [10]:
# 3️⃣ DEMO 3 · Step 1 · Translators: invoice <-> JSON / XML
import json
import xml.etree.ElementTree as ET

def to_json(record):  return json.dumps(record, ensure_ascii=False)
def from_json(text):  return json.loads(text)

def to_xml(record):
    root = ET.Element("invoice")
    for key, value in record.items():
        ET.SubElement(root, key).text = str(value)
    return ET.tostring(root, encoding="unicode")

def from_xml(text):
    record = {}
    for child in ET.fromstring(text):
        value = child.text or ""
        try:
            number = float(value)
            record[child.tag] = int(number) if number.is_integer() else number
        except ValueError:
            record[child.tag] = value
    return record

WRITE = {"JSON": to_json, "XML": to_xml}
READ  = {"JSON": from_json, "XML": from_xml}
print("Translators ready: JSON <-> XML ✅")

Translators ready: JSON <-> XML ✅


## Demo 3 · Step 2 · The agents and the negotiation rule
**What it does:**
- `FormatAgent` holds an agent's **name** and the formats it **speaks**, in order of preference.
- `negotiate` finds the formats both agents share and picks the sender's favourite among them (else the receiver's).
- `send` runs the 4 steps and prints a log line for each: what an auditor would want to see.

**Point out:** the fail-safe branch. If there's no common format, it **refuses to send** and logs why, instead of sending something the receiver can't read.

In [11]:
# 3️⃣ DEMO 3 · Step 2 · Agents that declare formats, negotiate, convert, deliver or fail safely
from dataclasses import dataclass

@dataclass
class FormatAgent:
    name: str
    speaks: list      # formats in order of preference, e.g. ["XML", "JSON"]

def negotiate(sender, receiver):
    shared = [f for f in sender.speaks if f in receiver.speaks]
    return shared[0] if shared else None

def send(sender, receiver, record, written_in):
    print(f"1. DECLARE    {sender.name} speaks {sender.speaks} | {receiver.name} speaks {receiver.speaks}")
    agreed = negotiate(sender, receiver)
    print(f"2. NEGOTIATE  agreed format: {agreed or 'NONE'}")
    if agreed is None:
        print("4. FAIL SAFE  no common format → not sent. Logged for the integration team.")
        return {"delivered": False, "reason": "no common format"}

    original = WRITE[written_in](record)
    if written_in != agreed:
        print(f"3. CONVERT    {written_in} → {agreed}")
        payload = WRITE[agreed](READ[written_in](original))
    else:
        print("3. CONVERT    not needed")
        payload = original
    print(f"   before ({written_in}): {original}")
    print(f"   after  ({agreed}): {payload}")
    print(f"4. DELIVER    sent to {receiver.name} as {agreed} ✅")
    return {"delivered": True, "format": agreed}

## Demo 3 · Step 3 · Scenario A: accounts system → GST e-invoice portal (convert)
**The setup:** the Accounts agent can write XML and JSON but prefers XML (that's how its export works). The e-invoice agent only reads JSON.

**Point out:** they agree on JSON, the invoice is converted XML → JSON, and **the numbers survive the trip** (₹50,000 + 18% GST = ₹59,000).

In [12]:
# 3️⃣ DEMO 3 · Step 3 · Scenario A: XML invoice converted to JSON for the GST portal
invoice = {
    "invoice_no": "PS/2026-27/0142",
    "seller_gstin": "27ABCDE1234F1Z5",      # 27 = Maharashtra (sample GSTIN format)
    "buyer": "Shree Ganesh Traders, Pune",
    "taxable_value": 50000,
    "gst_rate": 18,
    "total": 59000,
}

accounts_agent = FormatAgent("AccountsAgent (ERP export)", ["XML", "JSON"])
einvoice_agent = FormatAgent("EInvoiceAgent (GST IRP)", ["JSON"])

send(accounts_agent, einvoice_agent, invoice, written_in="XML")

1. DECLARE    AccountsAgent (ERP export) speaks ['XML', 'JSON'] | EInvoiceAgent (GST IRP) speaks ['JSON']
2. NEGOTIATE  agreed format: JSON
3. CONVERT    XML → JSON
   before (XML): <invoice><invoice_no>PS/2026-27/0142</invoice_no><seller_gstin>27ABCDE1234F1Z5</seller_gstin><buyer>Shree Ganesh Traders, Pune</buyer><taxable_value>50000</taxable_value><gst_rate>18</gst_rate><total>59000</total></invoice>
   after  (JSON): {"invoice_no": "PS/2026-27/0142", "seller_gstin": "27ABCDE1234F1Z5", "buyer": "Shree Ganesh Traders, Pune", "taxable_value": 50000, "gst_rate": 18, "total": 59000}
4. DELIVER    sent to EInvoiceAgent (GST IRP) as JSON ✅


{'delivered': True, 'format': 'JSON'}

## Demo 3 · Step 4 · Scenario B: no common format (fail safely)
**The setup:** an old bank system only sends XML statements. The new WhatsApp alert service only reads JSON. Neither can translate.

**Point out:** the agent **refuses** and logs it. *"A bad payment file that's silently sent is far worse than one that's stopped with a clear reason."*

**Ask the room:** *"How would you fix this?"* → put a translator in the middle. That's Scenario C.

In [13]:
# 3️⃣ DEMO 3 · Step 4 · Scenario B: nothing in common → stop and log
legacy_bank_agent = FormatAgent("LegacyBankAgent (old core banking)", ["XML"])
whatsapp_agent    = FormatAgent("WhatsAppAlertAgent", ["JSON"])

send(legacy_bank_agent, whatsapp_agent, invoice, written_in="XML")

1. DECLARE    LegacyBankAgent (old core banking) speaks ['XML'] | WhatsAppAlertAgent speaks ['JSON']
2. NEGOTIATE  agreed format: NONE
4. FAIL SAFE  no common format → not sent. Logged for the integration team.


{'delivered': False, 'reason': 'no common format'}

## Demo 3 · Step 5 · Scenario C: add a translator agent in the middle
**What it does:** a middle agent that speaks both. The bank sends XML to it; it forwards JSON to WhatsApp. This is how real companies connect old and new systems ("middleware").

**Say:** *"Same as hiring an interpreter for a meeting between a Tamil-speaking supplier and a German buyer."*

In [14]:
# 3️⃣ DEMO 3 · Step 5 · Scenario C: a translator agent bridges the gap
translator_agent = FormatAgent("TranslatorAgent (middleware)", ["JSON", "XML"])

print("── Hop 1: bank → translator (they share XML) ──")
send(legacy_bank_agent, translator_agent, invoice, written_in="XML")
print("\n── Hop 2: translator → WhatsApp (they share JSON; translator converts) ──")
send(translator_agent, whatsapp_agent, invoice, written_in="XML")

── Hop 1: bank → translator (they share XML) ──
1. DECLARE    LegacyBankAgent (old core banking) speaks ['XML'] | TranslatorAgent (middleware) speaks ['JSON', 'XML']
2. NEGOTIATE  agreed format: XML
3. CONVERT    not needed
   before (XML): <invoice><invoice_no>PS/2026-27/0142</invoice_no><seller_gstin>27ABCDE1234F1Z5</seller_gstin><buyer>Shree Ganesh Traders, Pune</buyer><taxable_value>50000</taxable_value><gst_rate>18</gst_rate><total>59000</total></invoice>
   after  (XML): <invoice><invoice_no>PS/2026-27/0142</invoice_no><seller_gstin>27ABCDE1234F1Z5</seller_gstin><buyer>Shree Ganesh Traders, Pune</buyer><taxable_value>50000</taxable_value><gst_rate>18</gst_rate><total>59000</total></invoice>
4. DELIVER    sent to TranslatorAgent (middleware) as XML ✅

── Hop 2: translator → WhatsApp (they share JSON; translator converts) ──
1. DECLARE    TranslatorAgent (middleware) speaks ['JSON', 'XML'] | WhatsAppAlertAgent speaks ['JSON']
2. NEGOTIATE  agreed format: JSON
3. CONVERT    XML → JS

{'delivered': True, 'format': 'JSON'}

### 3️⃣ Demo 3 wrap-up (say this)
*"Before agents talk, they agree on how to talk: declare, negotiate, convert, deliver, or fail safely. It's the same reason the GST portal publishes an exact JSON format, and why banks reject files in the wrong layout instead of guessing."*

---
## Clean-up (end of class)
Closes the connection to the model.

In [15]:
# 🅰️ SETUP · Clean-up: close the model connection
await model_client.close()
print("Connection closed ✅")

Connection closed ✅


## If something goes wrong in class
| You see | Do this |
|---|---|
| Setup 2 fails with 404 | Your lab may use the course route: swap in `azure_endpoint=GATEWAY` and `api_version="2024-10-21"` (commented in Setup 2). |
| 401 | Wrong or expired key in `.env`. |
| 400 mentioning `temperature` | Nothing in this notebook sets it; check you didn't add one. |
| 429 | Rate limit. Wait 30–60 seconds and re-run the cell. |
| Demo 2 stops at 8 messages without TERMINATE | The safety net worked. Re-run Step 4; the report cell still saves the last plan if there is one. |
| `RuntimeError: ... event loop` | You're running the code as a `.py` script. In a script, wrap it in `async def main()` and call `asyncio.run(main())`. |